
# Klike Data Science Challenge
## Parte 3 — Recommendations Engine

**Empresa:** Millai | Klike  
**Dataset:** `klike_challenge_dataset.csv`  
**Linguagem:** Python  
**Objetivo:** desenvolver um sistema de recomendações acionáveis e contextualizadas para otimização de criativos publicitários.

---

### Contextualização

Na Parte 1, foram investigadas as características das campanhas, a qualidade dos dados e as associações entre atributos dos criativos e métricas de desempenho.

Na Parte 2, foi desenvolvido um modelo de regressão Ridge reduzido, utilizando 10 características disponíveis antes da veiculação das campanhas para estimar o `klike_score`.

O modelo apresentou MAE de 6,685 e R² de 0,667 no conjunto de teste.

Nesta terceira etapa, o objetivo é transformar os resultados exploratórios e preditivos em um **Recommendations Engine**, capaz de analisar uma campanha e identificar oportunidades de melhoria.

O sistema utilizará uma abordagem híbrida, combinando:

- **Modelagem preditiva:** simulação de alterações nas características dos criativos por meio do Ridge reduzido.
- **Evidências estatísticas:** comparação de campanhas com características semelhantes, considerando seu contexto.
- **Priorização:** ordenação das sugestões conforme as diferenças previstas no Klike Score, respeitando a suficiência das evidências disponíveis.

As recomendações serão interpretadas como oportunidades de otimização, e não como garantias de melhoria. Diferenças preditivas e associações observacionais não constituem evidências de causalidade.



### Metodologia do Recommendations Engine

O desenvolvimento será organizado nas seguintes etapas:

1. **Preparação e reprodutibilidade:** carregamento do dataset e recuperação do Ridge reduzido desenvolvido na Parte 2.
2. **Validação das entradas:** identificação das características necessárias para analisar uma campanha e verificação de sua consistência.
3. **Geração de cenários alternativos:** definição de alterações acionáveis nas características dos criativos.
4. **Simulação preditiva:** utilização do Ridge para estimar diferenças no Klike Score entre a campanha original e os cenários alternativos.
5. **Análise contextual:** investigação das associações observadas entre características criativas e desempenho em campanhas comparáveis.
6. **Seleção e priorização:** filtragem de alterações não sustentadas pelas evidências e ordenação das oportunidades identificadas.
7. **Geração das recomendações:** apresentação de sugestões específicas, quantificadas e acompanhadas de suas limitações.
8. **Demonstração e validação:** execução do sistema em campanhas do dataset e verificação da consistência dos resultados.

**Princípios metodológicos:**

- Reutilizar o modelo selecionado na Parte 2, evitando treinamento desnecessário.
- Utilizar exclusivamente os registros de treinamento para construir as evidências estatísticas que fundamentam as recomendações.
- Preservar o conjunto de teste, evitando utilizá-lo na definição ou otimização das regras do engine.
- Não utilizar métricas pós-veiculação como entradas do modelo preditivo.
- Diferenciar explicitamente associações estatísticas, diferenças entre previsões e efeitos causais.
- Evitar recomendações baseadas em grupos com suporte amostral insuficiente.
- Não apresentar melhorias quantitativas sem uma estimativa calculada e devidamente contextualizada.

O sistema será desenvolvido inicialmente como um protótipo funcional em Python, utilizando funções independentes e reproduzíveis.



---

## 1. Preparação do ambiente e recuperação do modelo

### 1.1. Importação das bibliotecas e verificação dos arquivos

Nesta etapa, serão importadas as bibliotecas necessárias para manipulação dos dados, recuperação do modelo e reprodução da divisão entre treinamento e teste.

O Ridge reduzido foi salvo na Parte 2 utilizando `joblib`, juntamente com a lista ordenada das features utilizadas no treinamento.

Antes de carregar os arquivos, será verificada sua disponibilidade no diretório do projeto.

Essa abordagem permite reutilizar o pipeline previamente treinado sem realizar novos ajustes nos parâmetros do modelo.


In [1]:

from pathlib import Path

import joblib
import pandas as pd

from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge

RANDOM_STATE = 42

# Caminhos dos arquivos
DATA_PATH = Path("klike_challenge_dataset.csv")
MODEL_PATH = Path("models/ridge_klike.joblib")
FEATURES_PATH = Path("models/features_klike.joblib")

# Verificação de disponibilidade
arquivos = {
    "Dataset original": DATA_PATH,
    "Ridge reduzido": MODEL_PATH,
    "Features do modelo": FEATURES_PATH
}

status_arquivos = pd.DataFrame([
    {
        "Arquivo": nome,
        "Caminho": str(caminho),
        "Disponível": caminho.is_file()
    }
    for nome, caminho in arquivos.items()
])

display(status_arquivos)


,Arquivo,Caminho,Disponível
0,Dataset original,klike_challenge_dataset.csv,True
1,Ridge reduzido,models\ridge_klike.joblib,True
2,Features do modelo,models\features_klike.joblib,True



### 1.2. Carregamento dos dados e reconstrução da divisão treino/teste

O dataset será carregado sem alterações nos registros originais.

Em seguida, serão recuperados o pipeline Ridge treinado e a lista de features utilizada na Parte 2.

Para preservar a reprodutibilidade, será reconstruída a mesma divisão aleatória entre treinamento e teste, utilizando `test_size=0.20` e `random_state=42`.

As variáveis booleanas serão convertidas para a representação numérica adotada anteriormente, preservando os valores ausentes para tratamento pelo próprio pipeline.

A reconstrução da divisão é necessária porque as estatísticas contextuais do Recommendations Engine serão calculadas exclusivamente sobre as campanhas de treinamento.

O conjunto de teste permanecerá separado e não será utilizado na elaboração das recomendações.


In [2]:

# Verificar os arquivos necessários
arquivos_ausentes = [
    str(caminho)
    for caminho in arquivos.values()
    if not caminho.is_file()
]

if arquivos_ausentes:
    raise FileNotFoundError(
        f"Arquivos não encontrados: {arquivos_ausentes}"
    )

# Carregar dataset e artefatos da Parte 2
df = pd.read_csv(DATA_PATH)

modelo_ridge = joblib.load(MODEL_PATH)
features_reduzidas = joblib.load(FEATURES_PATH)

# Features definidas na Parte 2
features_esperadas = [
    "platform",
    "category",
    "is_retargeting",
    "video_duration_s",
    "format",
    "has_subtitle",
    "has_cta",
    "has_hook",
    "has_face",
    "text_density"
]

assert features_reduzidas == features_esperadas, (
    "As features carregadas diferem da Parte 2."
)

assert isinstance(modelo_ridge, Pipeline), (
    "O modelo carregado não é um Pipeline."
)

assert isinstance(modelo_ridge.named_steps["model"], Ridge), (
    "O estimador carregado não é um Ridge."
)

assert modelo_ridge.named_steps["model"].alpha == 1.0, (
    "O alpha do Ridge difere da Parte 2."
)

assert df["klike_score"].notna().all(), (
    "O dataset contém valores ausentes no target."
)

assert df["campaign_id"].is_unique, (
    "Existem identificadores de campanha duplicados."
)

# Preparar as mesmas entradas do Ridge reduzido
X = df[features_reduzidas].copy()
y = df["klike_score"].copy()

colunas_booleanas = [
    "is_retargeting",
    "has_subtitle",
    "has_cta",
    "has_hook",
    "has_face"
]

for coluna in colunas_booleanas:
    X[coluna] = X[coluna].map({
        True: 1,
        False: 0
    }).astype(float)

# Reproduzir a divisão utilizada na Parte 2
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE
)

# Registros originais de treinamento para análises contextuais
df_train = df.loc[X_train.index].copy()

# Resumo da preparação
print("Modelo:", type(modelo_ridge.named_steps["model"]).__name__)
print("Alpha:", modelo_ridge.named_steps["model"].alpha)
print("Etapas do pipeline:", list(modelo_ridge.named_steps.keys()))

print("\nDimensões:")
print(f"Dataset original: {df.shape}")
print(f"Treinamento: {X_train.shape}")
print(f"Teste: {X_test.shape}")
print(f"Features utilizadas: {len(features_reduzidas)}")

print("\nFeatures do modelo:")
for i, feature in enumerate(features_reduzidas, start=1):
    print(f"{i:2d}. {feature}")

print("\nValidação dos índices:")
print(
    "Interseção treino/teste:",
    len(set(X_train.index) & set(X_test.index))
)
print(
    "Registros para evidência contextual:",
    len(df_train)
)


Modelo: Ridge
Alpha: 1.0
Etapas do pipeline: ['preprocessor', 'model']

Dimensões:
Dataset original: (500, 26)
Treinamento: (400, 10)
Teste: (100, 10)
Features utilizadas: 10

Features do modelo:
 1. platform
 2. category
 3. is_retargeting
 4. video_duration_s
 5. format
 6. has_subtitle
 7. has_cta
 8. has_hook
 9. has_face
10. text_density

Validação dos índices:
Interseção treino/teste: 0
Registros para evidência contextual: 400



---

## 2. Validação das entradas e definição das características modificáveis

### 2.1. Classificação das features

O Recommendations Engine utilizará as mesmas 10 features do Ridge reduzido, divididas em dois grupos:

**Características de contexto:** descrevem o ambiente e o direcionamento da campanha. Serão mantidas constantes durante as simulações para que as recomendações sejam contextualizadas.

**Características modificáveis:** representam aspectos do criativo que podem ser alterados para produzir cenários alternativos.

A classificação é importante para impedir que o sistema recomende alterações em características que não correspondem diretamente à otimização do criativo.

Nesta etapa, também serão verificados os valores categóricos presentes no conjunto de treinamento e a quantidade de registros ausentes.

As categorias conhecidas serão utilizadas para validar os inputs. Essa validação evita que valores desconhecidos sejam silenciosamente codificados pelo pipeline, comprometendo a interpretação das previsões.


In [3]:

# Características que representam o contexto da campanha
features_contexto = [
    "platform",
    "category",
    "is_retargeting"
]

# Características que podem ser modificadas no criativo
features_modificaveis = [
    "video_duration_s",
    "format",
    "has_subtitle",
    "has_cta",
    "has_hook",
    "has_face",
    "text_density"
]

# Classificação por tipo de dado
features_categoricas = [
    "platform",
    "category",
    "format",
    "text_density"
]

features_booleanas = [
    "is_retargeting",
    "has_subtitle",
    "has_cta",
    "has_hook",
    "has_face"
]

features_numericas = [
    "video_duration_s"
]

# Verificar se todas as features foram classificadas
assert set(features_contexto).isdisjoint(
    features_modificaveis
)

assert set(features_contexto + features_modificaveis) == set(
    features_reduzidas
)

assert set(
    features_categoricas + features_booleanas + features_numericas
) == set(features_reduzidas)

# Resumo das características
resumo_features = pd.DataFrame({
    "Feature": features_reduzidas,
    "Papel": [
        "Contexto" if feature in features_contexto
        else "Modificável"
        for feature in features_reduzidas
    ],
    "Tipo": [
        "Categórica" if feature in features_categoricas
        else "Booleana" if feature in features_booleanas
        else "Numérica"
        for feature in features_reduzidas
    ],
    "Valores ausentes no treino": [
        df_train[feature].isna().sum()
        for feature in features_reduzidas
    ]
})

display(resumo_features)

# Categorias efetivamente observadas no treinamento
categorias_validas = {
    feature: sorted(df_train[feature].dropna().unique().tolist())
    for feature in features_categoricas
}

print("\nCategorias disponíveis no treinamento:")

for feature, categorias in categorias_validas.items():
    print(f"{feature}: {categorias}")

print("\nDistribuição da duração do vídeo no treinamento:")
display(
    df_train["video_duration_s"]
    .describe()
    .to_frame(name="video_duration_s")
)


,Feature,Papel,Tipo,Valores ausentes no treino
0,platform,Contexto,Categórica,0
1,category,Contexto,Categórica,0
2,is_retargeting,Contexto,Booleana,0
3,video_duration_s,Modificável,Numérica,34
4,format,Modificável,Categórica,0
5,has_subtitle,Modificável,Booleana,38
6,has_cta,Modificável,Booleana,0
7,has_hook,Modificável,Booleana,0
8,has_face,Modificável,Booleana,0
9,text_density,Modificável,Categórica,0



Categorias disponíveis no treinamento:
platform: ['LinkedIn', 'Meta', 'TikTok']
category: ['App Install', 'Branding', 'E-commerce', 'Lead Gen', 'SaaS']
format: ['horizontal', 'quadrado', 'vertical']
text_density: ['high', 'low', 'medium']

Distribuição da duração do vídeo no treinamento:


,video_duration_s
count,366.000000
mean,19.702186
std,14.168704
min,5.000000
25%,11.000000
50%,16.000000
75%,26.000000
max,115.000000



### 2.2. Preparação e validação de uma campanha individual

O protótipo receberá uma campanha representada por um dicionário Python ou por uma linha do dataset (`pandas.Series`).

A função de preparação terá as seguintes responsabilidades:

- Verificar se as 10 features necessárias estão presentes.
- Validar os valores categóricos utilizando apenas as categorias observadas no treinamento.
- Converter características booleanas para a representação numérica utilizada no Ridge.
- Verificar se a duração do vídeo é numericamente válida.
- Preservar valores ausentes nas características que já possuíam ausências no dataset e foram tratadas pelo pipeline.
- Organizar as características na ordem esperada pelo modelo.

O processo não utilizará `klike_score`, CTR, ROAS ou outras métricas pós-veiculação como entradas preditivas.

A função desenvolvida nesta seção servirá como interface de entrada para o Recommendations Engine, permitindo analisar campanhas existentes ou novas campanhas informadas manualmente.


In [4]:

import numpy as np

def preparar_campanha(campanha):
    """
    Valida e prepara uma campanha para o Ridge reduzido.

    Parâmetros
    ----------
    campanha : dict ou pd.Series
        Características de uma campanha.

    Retorna
    -------
    pd.DataFrame
        DataFrame com uma linha e as 10 features
        na ordem esperada pelo modelo.
    """

    if isinstance(campanha, pd.Series):
        campanha = campanha.to_dict()

    if not isinstance(campanha, dict):
        raise TypeError(
            "A campanha deve ser um dict ou pd.Series."
        )

    features_ausentes = [
        feature
        for feature in features_reduzidas
        if feature not in campanha
    ]

    if features_ausentes:
        raise ValueError(
            f"Features ausentes: {features_ausentes}"
        )

    entrada = {
        feature: campanha[feature]
        for feature in features_reduzidas
    }

    # Validar variáveis categóricas
    for feature in features_categoricas:
        valor = entrada[feature]

        if pd.isna(valor) or valor not in categorias_validas[feature]:
            raise ValueError(
                f"Valor inválido para {feature}: {valor}. "
                f"Valores aceitos: {categorias_validas[feature]}"
            )

    # Converter booleanos para 0 ou 1
    for feature in features_booleanas:
        valor = entrada[feature]

        if pd.isna(valor):
            if feature == "has_subtitle":
                entrada[feature] = np.nan
                continue

            raise ValueError(
                f"A feature {feature} não pode ser nula."
            )

        if isinstance(valor, (bool, np.bool_)):
            entrada[feature] = int(valor)

        elif isinstance(valor, (int, float, np.integer, np.floating)) \
                and valor in (0, 1):
            entrada[feature] = int(valor)

        else:
            raise ValueError(
                f"{feature} deve ser True, False, 0 ou 1."
            )

    # Validar a duração do vídeo
    duracao = entrada["video_duration_s"]

    if pd.isna(duracao):
        entrada["video_duration_s"] = np.nan

    elif (
        isinstance(duracao, (bool, np.bool_))
        or not isinstance(
            duracao,
            (int, float, np.integer, np.floating)
        )
        or not np.isfinite(duracao)
        or duracao <= 0
    ):
        raise ValueError(
            "video_duration_s deve ser um número positivo "
            "e finito, ou um valor ausente."
        )

    else:
        entrada["video_duration_s"] = float(duracao)

    return pd.DataFrame(
        [entrada],
        columns=features_reduzidas
    )


# Selecionar uma campanha do conjunto de treinamento
campanha_exemplo = df_train.iloc[0]

# Preparar o input
entrada_validada = preparar_campanha(campanha_exemplo)

print("Campanha selecionada:")
print(campanha_exemplo["campaign_id"])

print("\nInput validado para o Recommendations Engine:")
display(entrada_validada)


Campanha selecionada:
KLK-0250

Input validado para o Recommendations Engine:


,platform,category,is_retargeting,video_duration_s,format,has_subtitle,has_cta,has_hook,has_face,text_density
0,LinkedIn,App Install,0,22.0,vertical,1,1,1,0,medium



---

## 3. Geração de cenários alternativos

### 3.1. Definição das alterações candidatas

O Recommendations Engine avaliará possíveis modificações nas características dos criativos, preservando o contexto original de cada campanha.

Cada cenário será gerado alterando **uma única feature por vez**, permitindo comparar a previsão da campanha original com a previsão de uma configuração alternativa.

As modificações serão definidas conforme o tipo de variável:

- **Booleanas:** testar a inclusão de características ausentes, como hook, CTA, legendas e rosto humano.
- **Categóricas:** testar formatos e densidades de texto existentes no conjunto de treinamento.
- **Numéricas:** testar durações alternativas baseadas nos quartis observados em campanhas da mesma plataforma, sem assumir que esses valores sejam ideais.

As características `platform`, `category` e `is_retargeting` permanecerão constantes durante as simulações.

Valores ausentes não serão interpretados automaticamente como ausência de uma característica. Nesses casos, não serão geradas alterações para a feature até que seu valor original seja conhecido.

Os cenários representam **hipóteses de melhoria**, não recomendações finais. Sua seleção dependerá posteriormente das diferenças previstas pelo Ridge e das evidências estatísticas contextuais.

Para evitar data leakage, as alternativas baseadas na distribuição dos dados serão calculadas exclusivamente a partir do conjunto de treinamento.


In [5]:

def gerar_cenarios(campanha):
    """
    Gera cenários alternativos modificando uma feature
    por vez, mantendo o contexto original da campanha.

    Não calcula impacto nem classifica melhorias.
    """

    entrada = preparar_campanha(campanha)
    original = entrada.iloc[0].to_dict()

    cenarios = []

    def adicionar_cenario(feature, novo_valor, origem):
        alternativa = original.copy()
        alternativa[feature] = novo_valor

        cenarios.append({
            "feature_alterada": feature,
            "valor_original": original[feature],
            "valor_proposto": novo_valor,
            "origem": origem,
            **alternativa
        })

    # 1. Características booleanas
    booleanas_modificaveis = [
        "has_subtitle",
        "has_cta",
        "has_hook",
        "has_face"
    ]

    for feature in booleanas_modificaveis:
        valor = original[feature]

        if pd.notna(valor) and valor == 0:
            adicionar_cenario(
                feature,
                1,
                "Inclusão de característica ausente"
            )

    # 2. Características categóricas
    categoricas_modificaveis = [
        "format",
        "text_density"
    ]

    for feature in categoricas_modificaveis:
        for valor in categorias_validas[feature]:

            if valor != original[feature]:
                adicionar_cenario(
                    feature,
                    valor,
                    "Categoria observada no treinamento"
                )

    # 3. Duração do vídeo
    duracao_atual = original["video_duration_s"]

    if pd.notna(duracao_atual):

        duracoes_plataforma = df_train.loc[
            df_train["platform"] == original["platform"],
            "video_duration_s"
        ].dropna()

        # Evitar referências baseadas em grupos muito pequenos
        if len(duracoes_plataforma) >= 20:

            quartis = duracoes_plataforma.quantile(
                [0.25, 0.50, 0.75]
            )

            duracoes_candidatas = sorted({
                int(round(valor))
                for valor in quartis
                if valor > 0
            })

            for duracao in duracoes_candidatas:

                if not np.isclose(duracao, duracao_atual):
                    adicionar_cenario(
                        "video_duration_s",
                        duracao,
                        "Quartis da duração por plataforma no treino"
                    )

    # Organizar os cenários
    colunas = [
        "feature_alterada",
        "valor_original",
        "valor_proposto",
        "origem"
    ] + features_reduzidas

    return pd.DataFrame(cenarios, columns=colunas)


# Gerar cenários para a campanha de exemplo
cenarios_exemplo = gerar_cenarios(campanha_exemplo)

print("Campanha:", campanha_exemplo["campaign_id"])
print("Total de cenários:", len(cenarios_exemplo))

print("\nAlterações candidatas:")
display(
    cenarios_exemplo[
        [
            "feature_alterada",
            "valor_original",
            "valor_proposto",
            "origem"
        ]
    ]
)

print("\nQuantidade de cenários por feature:")
display(
    cenarios_exemplo["feature_alterada"]
    .value_counts()
    .rename_axis("Feature")
    .reset_index(name="Cenários")
)


Campanha: KLK-0250
Total de cenários: 8

Alterações candidatas:


,feature_alterada,valor_original,valor_proposto,origem
0,has_face,0,1,Inclusão de característica ausente
1,format,vertical,horizontal,Categoria observada no treinamento
2,format,vertical,quadrado,Categoria observada no treinamento
3,text_density,medium,high,Categoria observada no treinamento
4,text_density,medium,low,Categoria observada no treinamento
5,video_duration_s,22.0,18,Quartis da duração por plataforma no treino
6,video_duration_s,22.0,26,Quartis da duração por plataforma no treino
7,video_duration_s,22.0,31,Quartis da duração por plataforma no treino



Quantidade de cenários por feature:


,Feature,Cenários
0,video_duration_s,3
1,format,2
2,text_density,2
3,has_face,1


In [6]:

def valores_iguais(a, b):
    if pd.isna(a) and pd.isna(b):
        return True

    return a == b


def validar_cenarios(campanha, cenarios):
    original = preparar_campanha(campanha).iloc[0]

    for _, cenario in cenarios.iterrows():

        # Identificar todas as features alteradas
        alteradas = [
            feature
            for feature in features_reduzidas
            if not valores_iguais(
                original[feature],
                cenario[feature]
            )
        ]

        # Exatamente uma feature deve ser modificada
        assert len(alteradas) == 1, (
            f"Cenário inválido: {alteradas}"
        )

        assert alteradas[0] == cenario["feature_alterada"]

        # Preservar o contexto
        for feature in features_contexto:
            assert valores_iguais(
                original[feature],
                cenario[feature]
            )

        # Garantir que o cenário seja aceito pelo input
        preparar_campanha(
            cenario[features_reduzidas]
        )

    print("Todos os cenários passaram na validação.")
    print("Contexto preservado e apenas uma alteração por cenário.")


validar_cenarios(
    campanha_exemplo,
    cenarios_exemplo
)


Todos os cenários passaram na validação.
Contexto preservado e apenas uma alteração por cenário.
